# Aria Security & Compliance Guide

**Complete guide for security best practices, compliance requirements, threat modeling, and secure development patterns.**

Learn how to build secure Aria features, protect user data, and maintain compliance.


## Security Overview

### Security Principles

```
1. Defense in Depth
   Multiple layers of security controls

2. Least Privilege
   Users/services get minimum required permissions

3. Secure by Default
   Safe defaults, opt-in for risky operations

4. Zero Trust
   Verify everything, trust nothing

5. Privacy by Design
   User data protection built in from start
```

### Common Threats & Mitigations

| Threat                                | Impact                          | Mitigation                                 |
| ------------------------------------- | ------------------------------- | ------------------------------------------ |
| **SQL Injection**                     | Data theft, deletion            | Parameterized queries, ORM usage           |
| **XSS (Cross-Site Scripting)**        | Session theft, malware          | Input sanitization, CSP headers            |
| **CSRF (Cross-Site Request Forgery)** | Unwanted actions                | CSRF tokens, SameSite cookies              |
| **Weak Authentication**               | Account takeover                | MFA, rate limiting, secure storage         |
| **Data Breaches**                     | Privacy violation, reputation   | Encryption, access controls, auditing      |
| **DoS (Denial of Service)**           | Service unavailable             | Rate limiting, DDoS protection, monitoring |
| **Insecure Dependencies**             | Malicious code, vulnerabilities | Dependency scanning, version pinning       |
| **Configuration Leaks**               | Secret exposure                 | Env vars, vault, .gitignore                |

---

## Authentication & Authorization

### User Authentication

**Best Practices:**

```python
# 1. Use secure password hashing
from werkzeug.security import generate_password_hash, check_password_hash

# Hash password on registration
hashed = generate_password_hash(password, method='pbkdf2:sha256', salt_length=16)
user.password_hash = hashed

# Verify on login
if check_password_hash(user.password_hash, provided_password):
    # Login successful
    pass

# 2. Enforce strong password requirements
PASSWORD_RULES = {
    'min_length': 12,
    'require_uppercase': True,
    'require_numbers': True,
    'require_special': True,
}

def validate_password(password: str) -> tuple[bool, str]:
    """Validate password strength."""
    if len(password) < PASSWORD_RULES['min_length']:
        return False, "Password too short"
    if not any(c.isupper() for c in password):
        return False, "Need uppercase letter"
    if not any(c.isdigit() for c in password):
        return False, "Need number"
    if not any(c in '!@#$%^&*' for c in password):
        return False, "Need special character"
    return True, "Valid"

# 3. Implement rate limiting
from flask_limiter import Limiter
from flask_limiter.util import get_remote_address

limiter = Limiter(
    key_func=get_remote_address,
    default_limits=["200 per day", "50 per hour"]
)

@app.route('/login', methods=['POST'])
@limiter.limit("5 per minute")
def login():
    # Rate limited to 5 attempts per minute
    pass
```

**Multi-Factor Authentication:**

```python
import pyotp

# Generate TOTP secret
secret = pyotp.random_base32()
user.mfa_secret = secret

# User scans QR code or enters secret in authenticator app
totp = pyotp.TOTP(secret)
provisioning_uri = totp.provisioning_uri(
    name=user.email,
    issuer_name='Aria'
)

# Verify TOTP code
def verify_mfa(code: str) -> bool:
    totp = pyotp.TOTP(user.mfa_secret)
    return totp.verify(code)
```

### Authorization & Access Control

**Role-Based Access Control (RBAC):**

```python
from enum import Enum
from functools import wraps

class Role(Enum):
    ADMIN = "admin"
    MODERATOR = "moderator"
    USER = "user"
    GUEST = "guest"

# Define permissions per role
PERMISSIONS = {
    Role.ADMIN: ["read", "write", "delete", "manage_users"],
    Role.MODERATOR: ["read", "write", "delete"],
    Role.USER: ["read", "write"],
    Role.GUEST: ["read"],
}

def require_permission(permission: str):
    """Decorator to enforce permissions."""
    def decorator(f):
        @wraps(f)
        def decorated_function(*args, **kwargs):
            user = current_user
            role = user.role

            if permission not in PERMISSIONS[role]:
                abort(403, "Insufficient permissions")

            return f(*args, **kwargs)
        return decorated_function
    return decorator

# Usage
@app.route('/api/admin/users', methods=['DELETE'])
@require_permission("manage_users")
def delete_user():
    # Only admins can access
    pass
```

**Resource-Level Access Control:**

```python
def can_access_resource(user: User, resource: Resource) -> bool:
    """Check if user can access resource."""
    # User owns resource
    if resource.owner_id == user.id:
        return True

    # User is admin
    if user.role == Role.ADMIN:
        return True

    # User is in shared group
    if resource.id in [g.resource_id for g in user.groups]:
        return True

    return False

@app.route('/api/resources/<int:resource_id>', methods=['GET'])
def get_resource(resource_id: int):
    resource = Resource.query.get(resource_id)

    if not can_access_resource(current_user, resource):
        abort(403, "Access denied")

    return jsonify(resource)
```

---

## Data Security

### Encryption at Rest

```python
from cryptography.fernet import Fernet

# Generate encryption key (store securely in vault)
encryption_key = Fernet.generate_key()
cipher = Fernet(encryption_key)

# Encrypt sensitive data
sensitive_data = "user_ssn_123_45_6789"
encrypted = cipher.encrypt(sensitive_data.encode())

# Store encrypted in database
user.ssn_encrypted = encrypted

# Decrypt when needed
decrypted = cipher.decrypt(user.ssn_encrypted).decode()
```

### Encryption in Transit

```python
# All APIs use HTTPS (TLS 1.3+)
# Configure in Azure App Service

# Set security headers
from flask import Flask

app = Flask(__name__)

@app.after_request
def set_security_headers(response):
    # Force HTTPS
    response.headers['Strict-Transport-Security'] = 'max-age=31536000; includeSubDomains'

    # Prevent clickjacking
    response.headers['X-Frame-Options'] = 'DENY'

    # Prevent MIME type sniffing
    response.headers['X-Content-Type-Options'] = 'nosniff'

    # Enable XSS protection
    response.headers['X-XSS-Protection'] = '1; mode=block'

    # Content Security Policy
    response.headers['Content-Security-Policy'] = "default-src 'self'; script-src 'self' 'unsafe-inline'; style-src 'self' 'unsafe-inline'"

    return response
```

### Data Retention & Deletion

```python
from datetime import datetime, timedelta

class UserData(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    user_id = db.Column(db.Integer)
    content = db.Column(db.String)
    created_at = db.Column(db.DateTime, default=datetime.utcnow)
    deleted_at = db.Column(db.DateTime, nullable=True)

def soft_delete_user_data(user_id: int):
    """Soft delete user data (keep for compliance)."""
    data = UserData.query.filter_by(user_id=user_id).all()
    for record in data:
        record.deleted_at = datetime.utcnow()
    db.session.commit()

def hard_delete_old_data():
    """Hard delete data older than 90 days."""
    cutoff_date = datetime.utcnow() - timedelta(days=90)
    UserData.query.filter(
        UserData.deleted_at < cutoff_date
    ).delete()
    db.session.commit()

# Schedule daily
from apscheduler.schedulers.background import BackgroundScheduler
scheduler = BackgroundScheduler()
scheduler.add_job(hard_delete_old_data, 'cron', hour=2, minute=0)
scheduler.start()
```

---

## Input Validation & Output Encoding

### Input Validation

```python
from pydantic import BaseModel, validator, EmailStr
import re

class UserInput(BaseModel):
    email: EmailStr  # Validates email format
    username: str
    bio: str
    age: int

    @validator('username')
    def username_valid(cls, v):
        # No special characters, 3-20 chars
        if not re.match(r'^[a-zA-Z0-9_]{3,20}$', v):
            raise ValueError('Invalid username format')
        return v

    @validator('bio')
    def bio_valid(cls, v):
        # No more than 500 chars
        if len(v) > 500:
            raise ValueError('Bio too long')
        return v

    @validator('age')
    def age_valid(cls, v):
        # Between 13 and 120
        if not 13 <= v <= 120:
            raise ValueError('Invalid age')
        return v

# Usage
@app.route('/api/users', methods=['POST'])
def create_user():
    data = UserInput(**request.json)
    # Only valid data reaches here
    user = User.create(data)
    return jsonify(user)
```

### Output Encoding

```python
from markupsafe import escape
from html import escape as html_escape

# Always escape user-generated content in HTML
@app.route('/posts/<int:post_id>')
def view_post(post_id: int):
    post = Post.query.get(post_id)

    # Escape content for HTML rendering
    safe_content = html_escape(post.content)

    return render_template('post.html', content=safe_content)

# Template (post.html) - use Jinja2 auto-escaping
<div class="post">
    {{ content }}  {# Automatically escaped #}
</div>
```

---

## API Security

### API Authentication

```python
from flask_httpauth import HTTPBearerAuth
import jwt
from datetime import datetime, timedelta

auth = HTTPBearerAuth()

# Generate JWT token
def generate_token(user_id: int) -> str:
    payload = {
        'user_id': user_id,
        'exp': datetime.utcnow() + timedelta(hours=24),
        'iat': datetime.utcnow()
    }
    token = jwt.encode(payload, app.config['SECRET_KEY'], algorithm='HS256')
    return token

# Verify token
@auth.verify_token
def verify_token(token: str) -> dict:
    try:
        payload = jwt.decode(token, app.config['SECRET_KEY'], algorithms=['HS256'])
        return payload
    except jwt.ExpiredSignatureError:
        return None
    except jwt.InvalidTokenError:
        return None

# Protect endpoints
@app.route('/api/protected')
@auth.login_required
def protected_endpoint():
    user_id = auth.current_user()['user_id']
    # Safe to use user_id
    return jsonify({'message': 'Protected data'})
```

### Rate Limiting & Throttling

```python
from flask_limiter import Limiter

limiter = Limiter(
    key_func=get_remote_address,
    storage_uri="redis://localhost:6379"
)

# Global rate limit
app.config['RATELIMIT_STORAGE_URL'] = 'redis://localhost:6379'

# Per-endpoint limits
@app.route('/api/chat', methods=['POST'])
@limiter.limit("10 per minute")  # 10 requests per minute
def chat():
    return jsonify({'response': 'Hello'})

@app.route('/api/admin/users', methods=['GET'])
@limiter.limit("100 per hour")  # 100 requests per hour
def list_users():
    return jsonify({'users': []})

# User-specific limits (authenticated)
@app.route('/api/quantum/submit', methods=['POST'])
@auth.login_required
@limiter.limit("5 per day;100 per month")  # 5 per day, 100 per month
def submit_quantum_job():
    return jsonify({'job_id': '123'})
```

---

## Secrets Management

### Environment Variables

```bash
# local.settings.json (development only, gitignored)
{
  "Values": {
    "AZURE_OPENAI_API_KEY": "your-key-here",
    "DATABASE_PASSWORD": "your-password",
    "JWT_SECRET": "your-secret"
  }
}

# DO NOT COMMIT THIS FILE
# DO NOT HARDCODE SECRETS IN CODE
```

**Good: Using environment variables**

```python
import os

API_KEY = os.environ.get('AZURE_OPENAI_API_KEY')
if not API_KEY:
    raise ValueError("AZURE_OPENAI_API_KEY not set")

# Use the key
response = openai.ChatCompletion.create(
    api_key=API_KEY,
    ...
)
```

**Bad: Hardcoded secrets**

```python
# NEVER DO THIS
API_KEY = "sk-1234567890abcdef"  # ← WRONG!
PASSWORD = "admin123"  # ← WRONG!
```

### Secret Rotation

```bash
# Update secrets in Azure Key Vault
az keyvault secret set \
  --vault-name aria-vault \
  --name AZURE-OPENAI-API-KEY \
  --value "new-key-value"

# Application automatically picks up new secret
# (with next restart or config refresh)
```

---

## Audit Logging & Monitoring

### Security Audit Logging

```python
from datetime import datetime
import json

class AuditLog(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    user_id = db.Column(db.Integer)
    action = db.Column(db.String)
    resource = db.Column(db.String)
    status = db.Column(db.String)  # success, failure
    details = db.Column(db.JSON)
    timestamp = db.Column(db.DateTime, default=datetime.utcnow)
    ip_address = db.Column(db.String)

def audit_log(
    user_id: int,
    action: str,
    resource: str,
    status: str,
    details: dict = None
):
    """Log security-relevant actions."""
    log = AuditLog(
        user_id=user_id,
        action=action,
        resource=resource,
        status=status,
        details=details or {},
        ip_address=request.remote_addr
    )
    db.session.add(log)
    db.session.commit()

# Log authentication attempts
@app.route('/login', methods=['POST'])
def login():
    email = request.json['email']
    password = request.json['password']

    user = User.query.filter_by(email=email).first()
    if not user or not verify_password(user, password):
        audit_log(
            user_id=None,
            action='login_failed',
            resource='user',
            status='failure',
            details={'email': email, 'reason': 'invalid_credentials'}
        )
        abort(401)

    audit_log(
        user_id=user.id,
        action='login_success',
        resource='user',
        status='success'
    )
    return jsonify({'token': generate_token(user.id)})
```

### Security Monitoring

```python
# Monitor for suspicious patterns
def check_security_alerts():
    """Check for security issues."""

    # Failed logins
    failed_logins = AuditLog.query.filter(
        AuditLog.action == 'login_failed',
        AuditLog.timestamp > datetime.utcnow() - timedelta(minutes=15)
    ).count()

    if failed_logins > 10:
        # Alert: potential brute force attack
        send_security_alert(f"High failed login rate: {failed_logins}")

    # Privilege escalation
    privilege_changes = AuditLog.query.filter(
        AuditLog.action == 'role_change',
        AuditLog.timestamp > datetime.utcnow() - timedelta(hours=1)
    ).count()

    if privilege_changes > 5:
        # Alert: unusual privilege changes
        send_security_alert(f"Multiple privilege changes: {privilege_changes}")
```

---

## Compliance Frameworks

### GDPR Compliance (User Data Rights)

```python
def export_user_data(user_id: int) -> dict:
    """Export all user data (GDPR right to portability)."""
    user = User.query.get(user_id)

    return {
        'personal_info': {
            'name': user.name,
            'email': user.email,
            'created_at': user.created_at
        },
        'chat_history': [
            {
                'timestamp': msg.timestamp,
                'content': msg.content
            }
            for msg in user.messages
        ],
        'settings': user.settings
    }

def delete_user_data(user_id: int):
    """Delete all user data (GDPR right to be forgotten)."""
    user = User.query.get(user_id)

    # Delete related data
    Message.query.filter_by(user_id=user_id).delete()
    Settings.query.filter_by(user_id=user_id).delete()

    # Soft delete user account
    user.deleted_at = datetime.utcnow()
    db.session.commit()
```

### Data Privacy Checklist

- [ ] Privacy policy written and accessible
- [ ] User consent collected for data processing
- [ ] Data retention policy defined
- [ ] User can export their data
- [ ] User can delete their data
- [ ] Audit logs maintained for 1+ years
- [ ] Data breach response plan documented
- [ ] GDPR/CCPA compliance verified

---

## Security Incident Response

### Incident Response Playbook

```
1. DETECT
   ├─ Monitor alerts
   ├─ Check logs
   └─ Identify affected systems

2. CONTAIN
   ├─ Isolate compromised systems
   ├─ Stop attack vector
   └─ Preserve evidence

3. ERADICATE
   ├─ Remove malicious code
   ├─ Patch vulnerabilities
   ├─ Update configurations
   └─ Verify removal

4. RECOVER
   ├─ Restore from backups
   ├─ Verify systems
   ├─ Monitor for recurrence
   └─ Document changes

5. LEARN
   ├─ Root cause analysis
   ├─ Identify preventions
   ├─ Update policies
   └─ Train team
```

### Example: Data Breach Response

```bash
#!/bin/bash
# Incident: Database credentials compromised

# 1. CONTAIN
# Revoke old credentials
az sql server ad-admin delete --resource-group aria-rg --server aria-server

# Take database offline (if severe)
# az sql db update --name aria_db --resource-group aria-rg --subscription default \
#   --set 'state=Paused'

# 2. PRESERVE EVIDENCE
# Copy logs for investigation
cp /var/log/audit.log /backup/incident_2026_07_05.log

# 3. ERADICATE
# Generate new credentials
NEW_PASSWORD=$(openssl rand -base64 32)
az sql server admin set-password --server aria-server --admin-user dbadmin --password "$NEW_PASSWORD"

# Update application secrets
az keyvault secret set --vault-name aria-vault --name DB-PASSWORD --value "$NEW_PASSWORD"

# 4. VERIFY
# Test database connection
sqlcmd -S aria-server.database.windows.net -U dbadmin -P "$NEW_PASSWORD" -Q "SELECT 1"

# 5. COMMUNICATE
echo "Database credentials rotated successfully"
```


## Security Testing & Validation

### Static Security Analysis

```bash
# Scan for common vulnerabilities
python -m bandit -r shared/ -f json -o data_out/bandit_report.json

# Check dependencies for known vulnerabilities
pip-audit --desc

# OWASP Dependency Check
dependency-check --project "Aria" --scan shared/
```

### Dynamic Security Testing

```bash
# SQL Injection testing
pytest tests/security/test_sql_injection.py -v

# XSS testing
pytest tests/security/test_xss_prevention.py -v

# CSRF testing
pytest tests/security/test_csrf_protection.py -v

# Authentication testing
pytest tests/security/test_authentication.py -v
```

### Security Checklist

- [ ] All passwords hashed with bcrypt/pbkdf2
- [ ] All APIs require authentication
- [ ] All user input validated
- [ ] All output properly encoded
- [ ] HTTPS enabled everywhere
- [ ] Security headers set
- [ ] Rate limiting enabled
- [ ] Logging and monitoring in place
- [ ] Dependencies scanned for vulnerabilities
- [ ] Security training completed
- [ ] Incident response plan documented
- [ ] Regular security audits scheduled
